In [1]:
from pynq import Overlay
from pynq import MMIO
from pynq import allocate
import numpy as np


In [2]:
ol = Overlay("/home/xilinx/overlays/axis_proj.bit")

In [3]:
help(ol)

Help on Overlay in module pynq.overlay:

<pynq.overlay.Overlay object>
    Default documentation for overlay /home/xilinx/overlays/axis_proj.bit. The following
    attributes are available on this overlay:
    
    IP Blocks
    ----------
    axi_dma_0            : pynq.lib.dma.DMA
    axi_gpio_0           : pynq.lib.axigpio.AxiGPIO
    processing_system7_0 : pynq.overlay.DefaultIP
    
    Hierarchies
    -----------
    None
    
    Interrupts
    ----------
    None
    
    GPIO Outputs
    ------------
    None
    
    Memories
    ------------
    PSDDR                : Memory



In [7]:
io = {
    "c1": ol.axi_gpio_0.channel1,
    "c2": ol.axi_gpio_0.channel2,
}
dma = ol.axi_dma_0
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel

In [8]:
# singel adder test
io["c1"].write(2, 0xFFFF)
io["c2"].write(8, 0xFFFF)


In [21]:
# Allocate input and output buffer in DRAM
buf_size = 32
input_buffer = allocate(shape=(buf_size,), dtype=np.uint32)
output_buffer = allocate(shape=(buf_size,), dtype=np.uint32)

In [26]:
# Write to input buffer
for i in range(buf_size):
    input_buffer[i] = i
    
# Check the written data
for i in range(buf_size):
    #print("0x%08X" % (input_buffer[i]))
    print("%d" % (input_buffer[i]))

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31


In [23]:
# Do AXI DMA MM2S transfer
dma_send.transfer(input_buffer)
# Do AXI DMA S2MM transfer
dma_recv.transfer(output_buffer)

In [25]:
# Check the memory content after DMA transfer
for i in range(buf_size):
    print("%d" % (output_buffer[i]))

10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41


In [19]:
# Delete buffer to prevent memory leak
del input_buffer, output_buffer